In [42]:
import pandas as pd
import geopandas as geo
import unicodedata


pd.options.display.float_format = '{:.6f}'.format

In [11]:
df = pd.read_csv(r"ijsn_map_uso_solo_es_2019_20200.csv", encoding='utf-8', usecols=['microestad', 'municipio'])

In [55]:
df.head()

,microestad,municipio
0,Centro-Oeste,Colatina
1,Centro-Oeste,Pancas
4,Centro-Oeste,Baixo Guandu
58,Noroeste,Ecoporanga
140,Noroeste,Barra de Sao Francisco


In [14]:
df = df.drop_duplicates()

In [58]:
df = df.fillna('Nordeste')

In [34]:
def remove_acentos(x):
    if pd.isna(x):
        return x
    s = str(x)
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

df['municipio'] = df['municipio'].apply(remove_acentos)

In [18]:
url_json = r"C:\Users\caleb\Downloads\PONTOS_1.zip"

In [19]:
df_pontos = geo.read_file(url_json, encoding='cp1252')

In [36]:
coordenadas_car = df_pontos['geometry'].copy()
coordenadas_car = coordenadas_car.get_coordinates()

In [37]:
car_sheet = pd.DataFrame(df_pontos).join(coordenadas_car)

In [38]:
car_sheet = car_sheet[['cod_imovel', 'municipio', 'x', 'y']]

In [40]:
car_sheet = car_sheet.merge(df, how='left', left_on='municipio', right_on='municipio')

In [44]:
car_sheet.head()

,cod_imovel,municipio,x,y,microestad
0,ES-3200102-FAD9FD9CD9FC40649A09800707279F44,Afonso Claudio,285613.707970,7772367.993289,Sudoeste Serrana
1,ES-3200102-5E36A0BEDE024C79A53E8C9ADD1C6109,Afonso Claudio,285132.038224,7765521.134571,Sudoeste Serrana
2,ES-3200102-9DCC8302C2624DF38414B055C73173A0,Afonso Claudio,279703.641989,7787421.052154,Sudoeste Serrana
3,ES-3200102-6A3925A7BCFF471F90B728C304FFAF11,Afonso Claudio,287046.664406,7778044.776864,Sudoeste Serrana
4,ES-3200102-9CD67FF0C725408EA7F6250F5D9241FA,Afonso Claudio,271985.908551,7773110.720386,Sudoeste Serrana


In [46]:
from sklearn.model_selection import train_test_split

In [60]:
RANDOM_STATE = 42
STRAT_COL = "microestad"

df_500, _ = train_test_split(
    car_sheet,
    train_size=500,
    stratify=car_sheet["microestad"],
    random_state=RANDOM_STATE
)

train, temp = train_test_split(
    df_500,
    test_size=0.40,
    stratify=df_500[STRAT_COL],
    random_state=RANDOM_STATE
)

val, test = train_test_split(
    temp,
    test_size=0.50,
    stratify=temp[STRAT_COL],
    random_state=RANDOM_STATE
)

print(len(train), len(val), len(test))

300 100 100


In [64]:
train = train[['cod_imovel', 'municipio', 'microestad', 'x', 'y']]
val = val[['cod_imovel', 'municipio', 'microestad', 'x', 'y']]
test = test[['cod_imovel', 'municipio', 'microestad', 'x', 'y']]

In [66]:
train

,cod_imovel,municipio,microestad,x,y
45809,ES-3202108-F48E24D447F040119F2A72AC81857F86,Ecoporanga,Noroeste,318550.434539,7957003.898176
4162,ES-3200102-AE8D329D041843D28D7CFE7941120F48,Afonso Claudio,Sudoeste Serrana,279153.047085,7777216.084058
105530,ES-3204559-7BE2775012034AF4A7543F4D04623195,Santa Maria de Jetiba,Central Serrana,321226.958740,7779043.993533
64029,ES-3203056-A281E98D1E6545D2BF0F604074ED7837,Jaguare,Nordeste,381990.780327,7911881.694139
96897,ES-3204351-258B06CC7AC24D7BAA142CFECD4B68FB,Rio Bananal,Rio Doce,364671.176784,7865488.565865
...,...,...,...,...,...
96398,ES-3204351-B6B53A2F5F3644D9AF166F859FCDA66F,Rio Bananal,Rio Doce,369636.234059,7859442.274112
126959,ES-3205176-85D34DE249024B09A4FC32513B9F2FEC,Vila Valerio,Centro-Oeste,358532.560138,7891129.788023
95029,ES-3204302-87C61FEAE98A4A2294DA6FFD59283033,Presidente Kennedy,Litoral Sul,293819.399275,7666318.411802
72444,ES-3203205-2B946B6077EA4B1C8C93E1E8FCACB996,Linhares,Rio Doce,353840.942443,7846510.122429


In [67]:
val

,cod_imovel,municipio,microestad,x,y
52006,ES-3202454-96BF5B1BCD564A11AB154469881B638F,Ibatiba,Caparaó,236374.502417,7762844.511030
108657,ES-3204609-20EEC119459644A88E590B865BD854BE,Santa Teresa,Central Serrana,327127.214822,7798107.858152
51527,ES-3202454-E222500886F1453CA123E0F9A6A30B14,Ibatiba,Caparaó,239156.101984,7760137.847653
103821,ES-3204559-8A92E3D4C9D04ECA8AC26990A329191F,Santa Maria de Jetiba,Central Serrana,316889.388967,7780616.095343
11373,ES-3200300-982FDD52C6FB469AB559EBBC163CB61C,Alfredo Chaves,Litoral Sul,295806.360165,7733600.704574
...,...,...,...,...,...
62232,ES-3203007-E6A708A149FD47B687E1D348DD11A701,Iuna,Caparaó,236771.094242,7752801.062577
119045,ES-3204955-8A8E33D2029C41B8BB4B9C66AAB8D9B9,Sao Roque do Canaa,Centro-Oeste,317984.588391,7826206.688244
52200,ES-3202454-C221C9A8A6BC427590D1F535A79DBCF2,Ibatiba,Caparaó,224768.477787,7759242.729273
98581,ES-3204401-358E4BC97BAA459E8DD5E7BADB61F2A7,Rio Novo do Sul,Litoral Sul,303914.914219,7690454.832758


In [65]:
test

,cod_imovel,municipio,microestad,x,y
80241,ES-3203502-82E3092110494428857501E7CFE14E7F,Montanha,Nordeste,360775.805962,8006406.673937
99432,ES-3204500-9CC4B82933F14C55A54830546C7329CD,Santa Leopoldina,Central Serrana,335540.643711,7776313.700423
36952,ES-3201803-7541B0853F1F493AA1AF1CB5FFBE86B6,Divino de Sao Lourenco,Caparaó,220973.021142,7720836.716749
89859,ES-3204005-1B50467AB5ED4177861696409CA3DF29,Pancas,Centro-Oeste,314820.848812,7871875.439826
92857,ES-3204104-1D4D03D2089B458189D0EC938C77A47D,Pinheiros,Nordeste,345131.813187,7975797.427978
...,...,...,...,...,...
124628,ES-3205150-48C81F57E41440DEAC15C475271F0284,Vila Pavao,Noroeste,324040.940405,7937491.083363
126213,ES-3205176-97A4FB0A15F0497E8509944DD995AFBA,Vila Valerio,Centro-Oeste,351259.376671,7902032.127264
128053,ES-3205176-C721A33EBA2B4F02AD280CEF97F428A5,Vila Valerio,Centro-Oeste,349884.926201,7898631.921434
22992,ES-3201001-6D06E92F9C624D6DB7BAEB749B515331,Boa Esperanca,Nordeste,352690.788442,7953116.402047


In [71]:
partitions = {
    "train": train,
    "val": val,
    "test": test
}

for parts, name in partitions.items():
    name.to_csv(f'{parts}_sample.csv', index=False, sep=';')